In [1]:
# ========================
# Week 5 - Day 4: Python Vulnerability Scanner
# ========================

In [2]:
import socket
import subprocess
import requests
import json
from datetime import datetime

def scan_ports(host, ports=[21,22,23,25,53,80,443,3306,8080,8888]):
    print(f"\n{'='*55}")
    print(f"🔍 PORT SCAN: {host}")
    print('='*55)
    
    open_ports = []
    
    for port in ports:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(1)
        result = sock.connect_ex((host, port))
        if result == 0:
            # Identify common services
            services = {
                21: "FTP", 22: "SSH", 23: "Telnet",
                25: "SMTP", 53: "DNS", 80: "HTTP",
                443: "HTTPS", 3306: "MySQL",
                8080: "HTTP-Alt", 8888: "Jupyter"
            }
            service = services.get(port, "Unknown")
            risk = "🔴 HIGH" if port in [21,23,3306] else "🟡 MEDIUM" if port in [22,25,8080,8888] else "🟢 LOW"
            print(f"Port {port:5d} ({service:10s}) → OPEN {risk}")
            open_ports.append({"port": port, "service": service})
        sock.close()
    
    if not open_ports:
        print("No common ports open ✅")
    
    return open_ports

In [3]:
# Test on localhost
ports_found = scan_ports("127.0.0.1")


🔍 PORT SCAN: 127.0.0.1
Port  8888 (Jupyter   ) → OPEN 🟡 MEDIUM


In [4]:
def analyze_headers(url):
    print(f"\n{'='*55}")
    print(f"🔍 SECURITY HEADERS: {url}")
    print('='*55)
    
    security_headers = {
        "Strict-Transport-Security": "🔴 Missing HSTS — HTTP downgrade possible",
        "X-Frame-Options":           "🔴 Missing — Clickjacking possible",
        "X-Content-Type-Options":    "🟡 Missing — MIME sniffing possible",
        "Content-Security-Policy":   "🔴 Missing — XSS attacks possible",
        "Referrer-Policy":           "🟡 Missing — Info leakage possible",
        "X-XSS-Protection":          "🟡 Missing — XSS protection disabled"
    }
    
    findings = []
    
    try:
        response = requests.get(url, timeout=5)
        headers = response.headers
        
        print(f"Status: {response.status_code}\n")
        
        for header, warning in security_headers.items():
            if header in headers:
                print(f"✅ {header}: {headers[header][:50]}")
            else:
                print(f"❌ {warning}")
                findings.append(warning)
                
    except Exception as e:
        print(f"Error: {e}")
    
    return findings

# Test on a real website
analyze_headers("https://github.com")


🔍 SECURITY HEADERS: https://github.com
Status: 200

✅ Strict-Transport-Security: max-age=31536000; includeSubdomains; preload
✅ X-Frame-Options: deny
✅ X-Content-Type-Options: nosniff
✅ Content-Security-Policy: default-src 'none'; base-uri 'self'; child-src git
✅ Referrer-Policy: origin-when-cross-origin, strict-origin-when-cross
✅ X-XSS-Protection: 0


[]

In [5]:
def analyze_headers(url):
    print(f"\n{'='*55}")
    print(f"🔍 SECURITY HEADERS: {url}")
    print('='*55)
    
    security_headers = {
        "Strict-Transport-Security": "🔴 Missing HSTS — HTTP downgrade possible",
        "X-Frame-Options":           "🔴 Missing — Clickjacking possible",
        "X-Content-Type-Options":    "🟡 Missing — MIME sniffing possible",
        "Content-Security-Policy":   "🔴 Missing — XSS attacks possible",
        "Referrer-Policy":           "🟡 Missing — Info leakage possible",
        "X-XSS-Protection":          "🟡 Missing — XSS protection disabled"
    }
    
    findings = []
    
    try:
        response = requests.get(url, timeout=5)
        headers = response.headers
        
        print(f"Status: {response.status_code}\n")
        
        for header, warning in security_headers.items():
            if header in headers:
                print(f"✅ {header}: {headers[header][:50]}")
            else:
                print(f"❌ {warning}")
                findings.append(warning)
                
    except Exception as e:
        print(f"Error: {e}")
    
    return findings

# Test on a real website
analyze_headers("https://mohamed-shafar-portfolio.vercel.app/")


🔍 SECURITY HEADERS: https://mohamed-shafar-portfolio.vercel.app/
Status: 200

✅ Strict-Transport-Security: max-age=63072000; includeSubDomains; preload
❌ 🔴 Missing — Clickjacking possible
❌ 🟡 Missing — MIME sniffing possible
❌ 🔴 Missing — XSS attacks possible
❌ 🟡 Missing — Info leakage possible
❌ 🟡 Missing — XSS protection disabled


['🔴 Missing — Clickjacking possible',
 '🟡 Missing — MIME sniffing possible',
 '🔴 Missing — XSS attacks possible',
 '🟡 Missing — Info leakage possible',
 '🟡 Missing — XSS protection disabled']

In [6]:
def test_sql_injection(url):
    print(f"\n{'='*55}")
    print(f"🔍 SQL INJECTION TEST: {url}")
    print('='*55)
    
    # Common SQL injection payloads
    payloads = [
        "' OR '1'='1",
        "' OR 1=1--",
        "'; DROP TABLE users--",
        "' UNION SELECT NULL--",
        "admin'--"
    ]
    
    # SQL error signatures
    error_signatures = [
        "mysql", "sqlite", "postgresql",
        "ORA-", "syntax error", "SQLSTATE",
        "microsoft", "odbc", "jdbc"
    ]
    
    vulnerabilities = []
    
    for payload in payloads:
        try:
            # Test in URL parameter
            test_url = f"{url}?id={payload}"
            response = requests.get(test_url, timeout=5)
            content = response.text.lower()
            
            for sig in error_signatures:
                if sig in content:
                    print(f"❌ VULNERABLE! Payload: {payload[:30]}")
                    print(f"   Error signature found: '{sig}'")
                    vulnerabilities.append(payload)
                    break
            else:
                print(f"✅ Safe against: {payload[:30]}")
                
        except Exception as e:
            print(f"⚠️  Error testing {payload[:20]}: {e}")
    
    return vulnerabilities

# Test on a deliberately vulnerable demo site
test_sql_injection("http://testphp.vulnweb.com/listproducts.php")


🔍 SQL INJECTION TEST: http://testphp.vulnweb.com/listproducts.php
⚠️  Error testing ' OR '1'='1: HTTPConnectionPool(host='testphp.vulnweb.com', port=80): Max retries exceeded with url: /listproducts.php?id='%20OR%20'1'='1 (Caused by ConnectTimeoutError(<HTTPConnection(host='testphp.vulnweb.com', port=80) at 0x7f791c2642f0>, 'Connection to testphp.vulnweb.com timed out. (connect timeout=5)'))
⚠️  Error testing ' OR 1=1--: HTTPConnectionPool(host='testphp.vulnweb.com', port=80): Max retries exceeded with url: /listproducts.php?id='%20OR%201=1-- (Caused by ConnectTimeoutError(<HTTPConnection(host='testphp.vulnweb.com', port=80) at 0x7f791c205090>, 'Connection to testphp.vulnweb.com timed out. (connect timeout=5)'))
⚠️  Error testing '; DROP TABLE users-: HTTPConnectionPool(host='testphp.vulnweb.com', port=80): Max retries exceeded with url: /listproducts.php?id=';%20DROP%20TABLE%20users-- (Caused by ConnectTimeoutError(<HTTPConnection(host='testphp.vulnweb.com', port=80) at 0x7f791c204b9

[]

In [7]:
from http.server import HTTPServer, BaseHTTPRequestHandler
from urllib.parse import urlparse, parse_qs
import threading

# Create a deliberately vulnerable local server
class VulnerableHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        parsed = urlparse(self.path)
        params = parse_qs(parsed.query)
        
        self.send_response(200)
        self.send_header('Content-type', 'text/html')
        self.end_headers()
        
        user_id = params.get('id', ['1'])[0]
        
        # Deliberately vulnerable — echoes SQL error
        if "'" in user_id or "OR" in user_id or "--" in user_id:
            response = f"""
            <html><body>
            mysql error: You have an error in your SQL syntax
            near '{user_id}' at line 1
            SQLSTATE[42000]
            </body></html>
            """
        else:
            response = f"<html><body>Product ID: {user_id}</body></html>"
            
        self.wfile.write(response.encode())
    
    def log_message(self, format, *args):
        pass  # Suppress server logs

# Start vulnerable server in background
server = HTTPServer(('127.0.0.1', 9999), VulnerableHandler)
thread = threading.Thread(target=server.serve_forever)
thread.daemon = True
thread.start()
print("Vulnerable test server running on http://127.0.0.1:9999")

# Now test it
test_sql_injection("http://127.0.0.1:9999")

Vulnerable test server running on http://127.0.0.1:9999

🔍 SQL INJECTION TEST: http://127.0.0.1:9999
❌ VULNERABLE! Payload: ' OR '1'='1
   Error signature found: 'mysql'
❌ VULNERABLE! Payload: ' OR 1=1--
   Error signature found: 'mysql'
❌ VULNERABLE! Payload: '; DROP TABLE users--
   Error signature found: 'mysql'
❌ VULNERABLE! Payload: ' UNION SELECT NULL--
   Error signature found: 'mysql'
❌ VULNERABLE! Payload: admin'--
   Error signature found: 'mysql'


["' OR '1'='1",
 "' OR 1=1--",
 "'; DROP TABLE users--",
 "' UNION SELECT NULL--",
 "admin'--"]

In [8]:
def generate_report(target_host, target_url):
    print(f"\n{'='*55}")
    print(f"🛡️  VULNERABILITY ASSESSMENT REPORT")
    print(f"{'='*55}")
    print(f"Target Host: {target_host}")
    print(f"Target URL:  {target_url}")
    print(f"Scan Time:   {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Scanner:     Quantyber Scanner v1.0 — Sharuk")
    
    # Run all modules
    open_ports   = scan_ports(target_host)
    header_risks = analyze_headers(target_url)
    sqli_risks   = test_sql_injection(target_url)
    
    # Risk scoring
    total_issues = len(open_ports) + len(header_risks) + len(sqli_risks)
    
    if total_issues == 0:
        risk_level = "🟢 LOW"
    elif total_issues <= 3:
        risk_level = "🟡 MEDIUM"
    elif total_issues <= 6:
        risk_level = "🔴 HIGH"
    else:
        risk_level = "🚨 CRITICAL"
    
    # Summary
    print(f"\n{'='*55}")
    print(f"📊 EXECUTIVE SUMMARY")
    print(f"{'='*55}")
    print(f"Open Ports Found:        {len(open_ports)}")
    print(f"Header Issues:           {len(header_risks)}")
    print(f"SQL Injection Points:    {len(sqli_risks)}")
    print(f"Total Issues:            {total_issues}")
    print(f"Overall Risk Level:      {risk_level}")
    
    # Recommendations
    print(f"\n{'='*55}")
    print(f"💡 RECOMMENDATIONS")
    print(f"{'='*55}")
    
    if open_ports:
        print("• Close unnecessary open ports")
        print("• Disable Telnet (port 23) — use SSH instead")
        print("• Restrict database ports (3306) to localhost only")
    
    if header_risks:
        print("• Add missing security headers to web server config")
        print("• Implement Content Security Policy (CSP)")
        print("• Enable HSTS for HTTPS enforcement")
    
    if sqli_risks:
        print("• Use parameterized queries — never raw SQL")
        print("• Sanitize and validate all user inputs")
        print("• Implement Web Application Firewall (WAF)")
    
    print(f"\n{'='*55}")
    print(f"✅ Report Complete")
    print(f"{'='*55}")

# Generate full report
generate_report("127.0.0.1", "http://127.0.0.1:9999")


🛡️  VULNERABILITY ASSESSMENT REPORT
Target Host: 127.0.0.1
Target URL:  http://127.0.0.1:9999
Scan Time:   2026-07-19 19:07:25
Scanner:     Quantyber Scanner v1.0 — Sharuk

🔍 PORT SCAN: 127.0.0.1
Port  8888 (Jupyter   ) → OPEN 🟡 MEDIUM

🔍 SECURITY HEADERS: http://127.0.0.1:9999
Status: 200

❌ 🔴 Missing HSTS — HTTP downgrade possible
❌ 🔴 Missing — Clickjacking possible
❌ 🟡 Missing — MIME sniffing possible
❌ 🔴 Missing — XSS attacks possible
❌ 🟡 Missing — Info leakage possible
❌ 🟡 Missing — XSS protection disabled

🔍 SQL INJECTION TEST: http://127.0.0.1:9999
❌ VULNERABLE! Payload: ' OR '1'='1
   Error signature found: 'mysql'
❌ VULNERABLE! Payload: ' OR 1=1--
   Error signature found: 'mysql'
❌ VULNERABLE! Payload: '; DROP TABLE users--
   Error signature found: 'mysql'
❌ VULNERABLE! Payload: ' UNION SELECT NULL--
   Error signature found: 'mysql'
❌ VULNERABLE! Payload: admin'--
   Error signature found: 'mysql'

📊 EXECUTIVE SUMMARY
Open Ports Found:        1
Header Issues:           6
S